In [43]:
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values

In [44]:
df = pd.read_csv('Penguin_Dataset.csv')

In [45]:
df

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,41.1,18.4,195.0,3886.0,Female
1,Gentoo,Biscoe,50.0,15.4,217.0,4303.0,Male
2,Gentoo,Biscoe,50.0,15.1,220.0,4954.0,Female
3,Chinstrap,Dream,48.7,19.0,196.0,3739.0,Male
4,Adelie,Dream,39.0,17.8,187.0,3607.0,Male
...,...,...,...,...,...,...,...
995,Adelie,Dream,41.5,20.5,182.0,3931.0,Male
996,Gentoo,Biscoe,46.6,14.7,224.0,5050.0,Female
997,Adelie,Dream,36.4,16.6,183.0,3572.0,Male
998,Gentoo,Biscoe,47.9,15.0,213.0,5414.0,Female


In [46]:
df.shape

(1000, 7)

In [47]:
df.tail()

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
995,Adelie,Dream,41.5,20.5,182.0,3931.0,Male
996,Gentoo,Biscoe,46.6,14.7,224.0,5050.0,Female
997,Adelie,Dream,36.4,16.6,183.0,3572.0,Male
998,Gentoo,Biscoe,47.9,15.0,213.0,5414.0,Female
999,Adelie,Dream,38.5,18.9,187.0,4316.0,Female


In [48]:
df.columns

Index(['species', 'island', 'culmen_length_mm', 'culmen_depth_mm',
       'flipper_length_mm', 'body_mass_g', 'sex'],
      dtype='str')

In [49]:
# lets find the missing columns

missing_count = df.isna().sum()

In [50]:
missing_count

species               0
island                0
culmen_length_mm     10
culmen_depth_mm      10
flipper_length_mm    10
body_mass_g          10
sex                  10
dtype: int64

In [51]:
missing_rows = df[df.isna().any(axis=1)]

missing_rows

In [52]:
missing_rows

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
36,Adelie,Torgersen,39.6,17.5,NaN,3605.0,Male
38,Chinstrap,Dream,47.4,16.2,185.0,3958.0,NaN
42,Adelie,Dream,NaN,18.9,187.0,3567.0,Male
51,Gentoo,Biscoe,47.2,16.4,NaN,4792.0,Female
149,Adelie,Torgersen,38.3,18.7,195.0,NaN,Male
150,Gentoo,Biscoe,45.9,NaN,218.0,4541.0,Male
154,Gentoo,Biscoe,46.2,14.6,221.0,4848.0,NaN
160,Adelie,Dream,38.2,NaN,195.0,3677.0,Female
170,Chinstrap,Dream,45.8,19.4,205.0,3695.0,NaN
192,Gentoo,Biscoe,47.6,16.4,213.0,NaN,Male


In [53]:
df = df.dropna()

In [54]:
df.describe()

,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g
count,951.000000,951.000000,951.000000,951.00000
mean,43.825026,17.324711,199.191377,4096.22082
std,5.326905,1.770504,13.183449,703.69578
min,32.400000,13.000000,175.000000,2830.00000
25%,39.000000,15.800000,189.000000,3579.50000
50%,43.900000,17.800000,195.000000,3880.00000
75%,48.300000,18.700000,211.000000,4581.00000
max,56.000000,21.400000,235.000000,6317.00000


In [76]:
missing_rows = df[df.isna().any(axis=1)]
missing_rows

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex


In [58]:
df

,species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,41.1,18.4,195.0,3886.0,Female
1,Gentoo,Biscoe,50.0,15.4,217.0,4303.0,Male
2,Gentoo,Biscoe,50.0,15.1,220.0,4954.0,Female
3,Chinstrap,Dream,48.7,19.0,196.0,3739.0,Male
4,Adelie,Dream,39.0,17.8,187.0,3607.0,Male
...,...,...,...,...,...,...,...
995,Adelie,Dream,41.5,20.5,182.0,3931.0,Male
996,Gentoo,Biscoe,46.6,14.7,224.0,5050.0,Female
997,Adelie,Dream,36.4,16.6,183.0,3572.0,Male
998,Gentoo,Biscoe,47.9,15.0,213.0,5414.0,Female


In [61]:
conn = psycopg2.connect(
    host ='',
    port = ,
    database = '',
    user = ''
)
cursor = conn.cursor()
print("Connected Successfully")

Connected Successfully


In [68]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS penguin_details(
        species VARCHAR(100),
        island VARCHAR(100),
        culmen_length_mm NUMERIC(10,2),	
        culmen_depth_mm	NUMERIC(10,2),
        flipper_length_mm NUMERIC(10,2),
        body_mass_g	NUMERIC(10,2),
        sex VARCHAR(10)
        
    )
""")
conn.commit()
print("Connected to postgreSQL and Created!")

Connected to postgreSQL and Created!


In [69]:
# conn.rollback()

In [72]:
insert_query = """
    INSERT INTO penguin_details
    (species,island,culmen_length_mm,culmen_depth_mm,flipper_length_mm,body_mass_g,sex
)VALUES %s
"""
try:
    rows = list(df.itertuples(index = False,name = None))
    execute_values(cursor,insert_query,rows)
    conn.commit()
    print(f"Inserted {len(rows)} rows")
except Exception as e:
    conn.rollback()
    print("Error:",e)

Inserted 951 rows


In [74]:
cursor.execute("SELECT * FROM penguin_details ")
print(cursor.fetchone())

('Adelie', 'Torgersen', Decimal('41.10'), Decimal('18.40'), Decimal('195.00'), Decimal('3886.00'), 'Female')
